# 32 · Initialization audit (CPU, ~10 min; no training, no evaluation)
Rebuilds every bottleneck model with each seed exactly as the training code does (`clear_session`, `set_seeds(seed)`, `build_p2_model`)
and compares its initial weights, layer by layer, with the circuit model of the same placement (`g_q` or `p_q`).
Question: do the controls start from the same weights as the circuit model in every layer they share (encoder, bottleneck, decoder)?
Read-only for code, data and runs. Writes only `results/p2_rev5/init_audit/`.

In [ ]:
# 1 · Drive, code
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json
import numpy as np, pandas as pd
PROJECT = '/content/drive/MyDrive/quantum-placement'
CODE = f'{PROJECT}/code'; OUT = f'{PROJECT}/results/p2_rev5/init_audit'; os.makedirs(OUT, exist_ok=True)
QSEG_DIR = CODE
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p_ in (QSEG_DIR, CODE):
    if p_ not in sys.path: sys.path.insert(0, p_)
import tensorflow as tf, qseg2, qseg2_rev4; qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT)
print('qseg package:', QSEG_DIR, '| TF', tf.__version__)

In [ ]:
# 2 · Build every model with every seed and compare initial weights with the circuit model of the same placement
SHAPE = (cfg.image_size, cfg.image_size, cfg.in_channels)
def init_weights(name, seed):
    tf.keras.backend.clear_session(); tf.keras.mixed_precision.set_global_policy('float32')
    qseg2.set_seeds(seed)
    m = qseg2.build_p2_model(name, SHAPE, cfg.n_qubits, cfg.n_q_layers)
    w = {l.name: [v.numpy().copy() for v in l.weights] for l in m.layers if l.weights}
    order = [l.name for l in m.layers if l.weights]
    del m; return w, order
rows = []
for seed in range(5):
    for pl in ('g', 'p'):
        ref, order = init_weights(f'{pl}_q', seed)
        for ctl in ('cl', 'id', 'tr'):
            w, _ = init_weights(f'{pl}_{ctl}', seed)
            shared = [n for n in order if n in w and len(w[n]) == len(ref[n]) and all(a.shape == b.shape for a, b in zip(w[n], ref[n]))]
            diff = {n: max(float(np.abs(a - b).max()) for a, b in zip(w[n], ref[n])) for n in shared}
            differing = [n for n in shared if diff[n] > 0]
            n_par = sum(int(a.size) for n in shared for a in ref[n]); n_diff = sum(int(a.size) for n in differing for a in ref[n])
            rows.append(dict(seed=seed, placement=pl, control=f'{pl}_{ctl}', circuit=f'{pl}_q', shared_layers=len(shared),
                             identical_layers=len(shared) - len(differing), shared_params=n_par, params_in_differing_layers=n_diff,
                             max_abs_diff=max(diff.values()) if diff else np.nan,
                             first_differing_layer=differing[0] if differing else '', differing_layers=';'.join(differing)))
        print('seed', seed, pl, 'done', flush=True)
A = pd.DataFrame(rows); A.to_csv(f'{OUT}/init_audit.csv', index=False)
print(A[['seed', 'control', 'shared_layers', 'identical_layers', 'params_in_differing_layers', 'max_abs_diff', 'first_differing_layer']].to_string(index=False))

In [ ]:
# 3 · Summary by control (all seeds)
S = A.groupby('control').agg(seeds=('seed', 'nunique'), identical_all_seeds=('params_in_differing_layers', lambda v: bool((v == 0).all())),
                             max_abs_diff=('max_abs_diff', 'max'), differing_layers=('differing_layers', lambda v: sorted(set(';'.join(v).split(';')) - {''})))
print(S.to_string()); S.to_csv(f'{OUT}/init_audit_summary.csv')
print('written', OUT, '| DONE')